In [ ]:
import time
import copy
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
from collections import Counter
from PIL import Image
from tqdm import tqdm

*setup*

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

raw_dataset = r"C:\Users\Sina_Si1\Desktop\Project Code\Phase 2\PlantVillage"  # original dataset
balanced_dataset = r"C:\Users\Sina_Si1\Desktop\Project Code\Phase 2\PV_Balanced"  # after augmentation
split_output = r"C:\Users\Sina_Si1\Desktop\Project Code\Phase 2\PV_Split"  # train/val/test split

*Count Class Samples*

In [ ]:
def plot_class_distribution(dataset_path, title):
    class_counts = {cls: len(os.listdir(os.path.join(dataset_path, cls)))
                    for cls in os.listdir(dataset_path)}
    plt.figure(figsize=(15, 5))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xticks(rotation=90)
    plt.title(title)
    plt.show()
    return class_counts

print("Original class distribution:")
orig_counts = plot_class_distribution(raw_dataset, "Original dataset distribution")

*Augmentation*

In [ ]:
print("Balanced class distribution:")
balanced_counts = plot_class_distribution(balanced_dataset, "Balanced dataset distribution")

In [ ]:
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.Resize((224, 224))
])

os.makedirs(balanced_dataset, exist_ok=True)
max_count = max(orig_counts.values())

for cls, count in orig_counts.items():
    src_dir = os.path.join(raw_dataset, cls)
    dst_dir = os.path.join(balanced_dataset, cls)
    os.makedirs(dst_dir, exist_ok=True)

    # Copy original
    for img in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, img), os.path.join(dst_dir, img))

    # Augment until balanced
    imgs = os.listdir(src_dir)
    idx = 0
    while len(os.listdir(dst_dir)) < max_count:
        img = Image.open(os.path.join(src_dir, imgs[idx % len(imgs)])).convert("RGB")
        img_aug = transform_aug(img)
        save_path = os.path.join(dst_dir, f"aug_{idx}.png")
        img_aug.save(save_path)
        idx += 1

print("Balanced class distribution:")
balanced_counts = plot_class_distribution(balanced_dataset, "Balanced dataset distribution")